# Interactive Visual Analytics — Folium

**IBM Data Science Capstone — SpaceX Falcon 9**

Repository: [https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX](https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX)

Direct notebook URL after upload:  
[https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX/blob/main/06_SpaceX_Folium.ipynb](https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX/blob/main/06_SpaceX_Folium.ipynb)

**Data integrity note:** This notebook uses the project CSV files stored in the same repository.
No rows, metrics, charts, or model scores are manually invented.

## Objective

Use Folium to:
- mark all launch sites,
- mark successful and failed launch records,
- inspect launch-site geography,
- provide a reusable great-circle distance function for proximity analysis.

The maps and distances below use coordinates from `spacex_launch_geo.csv`.

In [1]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

geo = pd.read_csv("spacex_launch_geo.csv")
print("Dataset shape:", geo.shape)
geo.head()

Dataset shape: (56, 13)


,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


In [2]:
site_summary = (
    geo.groupby("Launch Site")
    .agg(
        Latitude=("Lat", "first"),
        Longitude=("Long", "first"),
        Launches=("Flight Number", "count"),
        Successes=("class", "sum"),
    )
    .reset_index()
)
site_summary["SuccessRate"] = site_summary["Successes"] / site_summary["Launches"]
site_summary

,Launch Site,Latitude,Longitude,Launches,Successes,SuccessRate
0,CCAFS LC-40,28.562302,-80.577356,26,7,0.269231
1,CCAFS SLC-40,28.563197,-80.576820,7,3,0.428571
2,KSC LC-39A,28.573255,-80.646895,13,10,0.769231
3,VAFB SLC-4E,34.632834,-120.610745,10,4,0.400000


## Map 1 — All launch sites

In [3]:
center = [geo["Lat"].mean(), geo["Long"].mean()]
site_map = folium.Map(location=center, zoom_start=4)

for _, row in site_summary.iterrows():
    popup = (
        f"{row['Launch Site']}<br>"
        f"Launches: {int(row['Launches'])}<br>"
        f"Successes: {int(row['Successes'])}<br>"
        f"Success rate: {row['SuccessRate']:.1%}"
    )
    folium.Circle(
        [row["Latitude"], row["Longitude"]],
        radius=1200,
        color="black",
        fill=True,
        fill_opacity=0.15,
        popup=popup,
    ).add_to(site_map)
    folium.Marker(
        [row["Latitude"], row["Longitude"]],
        popup=popup,
        tooltip=row["Launch Site"],
    ).add_to(site_map)

site_map

## Map 2 — Successful and failed launch records

In [4]:
launch_map = folium.Map(location=center, zoom_start=4)
cluster = MarkerCluster().add_to(launch_map)

for _, row in geo.iterrows():
    outcome = "Success" if int(row["class"]) == 1 else "Failure"
    marker_color = "green" if int(row["class"]) == 1 else "red"
    folium.Marker(
        [row["Lat"], row["Long"]],
        popup=f"Flight {row['Flight Number']}<br>{row['Launch Site']}<br>{outcome}",
        icon=folium.Icon(color=marker_color),
    ).add_to(cluster)

launch_map

## Great-circle distance helper

This helper can be used after identifying a nearby coast, road, railway, or city
point on the interactive map. No landmark coordinates are guessed here.

In [5]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    earth_radius_km = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return earth_radius_km * c

first_site = site_summary.iloc[0]
sanity = calculate_distance(
    first_site["Latitude"], first_site["Longitude"],
    first_site["Latitude"], first_site["Longitude"]
)
print("Distance sanity check (same point, km):", sanity)

Distance sanity check (same point, km): 0.0


In [6]:
pairs = []
for i, a in site_summary.iterrows():
    for j, b in site_summary.iterrows():
        if j <= i:
            continue
        pairs.append({
            "Site A": a["Launch Site"],
            "Site B": b["Launch Site"],
            "Distance_km": calculate_distance(
                a["Latitude"], a["Longitude"],
                b["Latitude"], b["Longitude"]
            ),
        })

pd.DataFrame(pairs).sort_values("Distance_km")

,Site A,Site B,Distance_km
0,CCAFS LC-40,CCAFS SLC-40,0.112488
1,CCAFS LC-40,KSC LC-39A,6.899304
3,CCAFS SLC-40,KSC LC-39A,6.934099
5,KSC LC-39A,VAFB SLC-4E,3819.052794
2,CCAFS LC-40,VAFB SLC-4E,3825.840279
4,CCAFS SLC-40,VAFB SLC-4E,3825.854482
